# Arena Ranker — Kaggle GPU 训练与推理

本 notebook 在 Kaggle 提供的 **GPU 环境（T4 16GB / P100 16GB）** 中完成：

1. 安装依赖
2. 设置项目源码
3. 训练 `Qwen/Qwen3-Embedding-0.6B` + LoRA 偏好分类器
4. 推理并生成 `submission.csv`

> **前置准备**
>
> | 项目 | 操作 |
> |------|------|
> | **竞赛数据** | Notebook 右侧 → Add Input → 搜索并添加竞赛数据集 |
> | **GPU** | Settings → Accelerator → 选择 **GPU T4 ×2** 或 **GPU P100** |
> | **联网** | Settings → Internet → **On**（用于下载 HuggingFace 模型）|
>
> 如果不想联网下载模型，请参考最后一节「离线模式」，提前把模型上传为 Kaggle Model。


## 1. 安装依赖

Kaggle 环境已预装 PyTorch 和部分常用库，这里只需补装缺失的包。

In [ ]:
!pip install -q "transformers>=4.55.0,<5" "peft>=0.17.0" "scikit-learn>=1.5.0" "tqdm>=4.66.0" "pyyaml>=6.0.2"

## 2. 写入项目源码

将 `arena_ranker` 包的所有源文件写入 `/kaggle/working/arena_ranker/`，并添加到 `sys.path`。

> **替代方案**：你也可以把本仓库上传为 Kaggle Dataset，然后直接
> `!pip install /kaggle/input/<your-dataset-slug>/` 来安装，不需要这一步。

In [ ]:
import base64
from pathlib import Path

PKG_DIR = Path("/kaggle/working/arena_ranker")
PKG_DIR.mkdir(parents=True, exist_ok=True)

_FILES = {
    "__init__.py": "IiIiQXJlbmEgcHJlZmVyZW5jZSByYW5rZXIgcGFja2FnZS4iIiIK",
    "config.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgYXNkaWN0LCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgeWFtbAoKCkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgRGF0YUNvbmZpZzoKICAgIHRyYWluX3BhdGg6IHN0ciA9ICJ0cmFpbi5jc3YiCiAgICB0ZXN0X3BhdGg6IHN0ciA9ICJ0ZXN0LmNzdiIKICAgIHRleHRfbWF4X2NoYXJzOiBpbnQgPSA0MDAwCiAgICB2YWxpZGF0aW9uX3NpemU6IGZsb2F0ID0gMC4xCiAgICByYW5kb21fc3RhdGU6IGludCA9IDQyCgoKQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBNb2RlbENvbmZpZzoKICAgIG1vZGVsX25hbWU6IHN0ciA9ICJRd2VuL1F3ZW4zLUVtYmVkZGluZy0wLjZCIgogICAgY2FjaGVfZGlyOiBzdHIgfCBOb25lID0gTm9uZQogICAgbG9jYWxfZmlsZXNfb25seTogYm9vbCA9IEZhbHNlCiAgICBtYXhfbGVuZ3RoOiBpbnQgPSA1MTIKICAgIGRyb3BvdXQ6IGZsb2F0ID0gMC4xCiAgICBmcmVlemVfZW5jb2RlcjogYm9vbCA9IEZhbHNlCiAgICB1c2VfbG9yYTogYm9vbCA9IFRydWUKICAgIGxvcmFfcjogaW50ID0gMTYKICAgIGxvcmFfYWxwaGE6IGludCA9IDMyCiAgICBsb3JhX2Ryb3BvdXQ6IGZsb2F0ID0gMC4wNQogICAgbG9yYV9iaWFzOiBzdHIgPSAibm9uZSIKICAgIGxvcmFfdGFza190eXBlOiBzdHIgPSAiZmVhdHVyZV9leHRyYWN0aW9uIgogICAgbG9yYV90YXJnZXRfbW9kdWxlczogbGlzdFtzdHJdID0gZmllbGQoCiAgICAgICAgZGVmYXVsdF9mYWN0b3J5PWxhbWJkYTogWyJxX3Byb2oiLCAia19wcm9qIiwgInZfcHJvaiIsICJvX3Byb2oiXQogICAgKQoKCkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgVHJhaW5pbmdDb25maWc6CiAgICBvdXRwdXRfZGlyOiBzdHIgPSAiYXJ0aWZhY3RzL2RlZmF1bHQiCiAgICBsZWFybmluZ19yYXRlOiBmbG9hdCA9IDJlLTUKICAgIGNsYXNzaWZpZXJfbGVhcm5pbmdfcmF0ZTogZmxvYXQgPSAxZS00CiAgICB3ZWlnaHRfZGVjYXk6IGZsb2F0ID0gMC4wMQogICAgYmF0Y2hfc2l6ZTogaW50ID0gMQogICAgZXBvY2hzOiBpbnQgPSAxCiAgICBncmFkX2FjY3VtX3N0ZXBzOiBpbnQgPSA4CiAgICB3YXJtdXBfcmF0aW86IGZsb2F0ID0gMC4xCiAgICBudW1fd29ya2VyczogaW50ID0gMAogICAgc2VlZDogaW50ID0gNDIKICAgIGFtcDogYm9vbCA9IFRydWUKICAgIGdyYWRpZW50X2NoZWNrcG9pbnRpbmc6IGJvb2wgPSBUcnVlCiAgICBsb2dfZXZlcnk6IGludCA9IDUwCgoKQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBBcHBDb25maWc6CiAgICBkYXRhOiBEYXRhQ29uZmlnID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PURhdGFDb25maWcpCiAgICBtb2RlbDogTW9kZWxDb25maWcgPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9TW9kZWxDb25maWcpCiAgICB0cmFpbmluZzogVHJhaW5pbmdDb25maWcgPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9VHJhaW5pbmdDb25maWcpCgogICAgZGVmIHRvX2RpY3Qoc2VsZikgLT4gZGljdFtzdHIsIEFueV06CiAgICAgICAgcmV0dXJuIGFzZGljdChzZWxmKQoKICAgIGRlZiBzYXZlKHNlbGYsIHBhdGg6IHN0ciB8IFBhdGgpIC0+IE5vbmU6CiAgICAgICAgb3V0cHV0X3BhdGggPSBQYXRoKHBhdGgpCiAgICAgICAgb3V0cHV0X3BhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBvdXRwdXRfcGF0aC53cml0ZV90ZXh0KHlhbWwuc2FmZV9kdW1wKHNlbGYudG9fZGljdCgpLCBzb3J0X2tleXM9RmFsc2UpLCBlbmNvZGluZz0idXRmLTgiKQoKCmRlZiBsb2FkX2NvbmZpZyhwYXRoOiBzdHIgfCBQYXRoIHwgTm9uZSA9IE5vbmUpIC0+IEFwcENvbmZpZzoKICAgIGlmIHBhdGggaXMgTm9uZToKICAgICAgICByZXR1cm4gQXBwQ29uZmlnKCkKCiAgICByYXcgPSB5YW1sLnNhZmVfbG9hZChQYXRoKHBhdGgpLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHJldHVybiBBcHBDb25maWcoCiAgICAgICAgZGF0YT1EYXRhQ29uZmlnKCoqcmF3LmdldCgiZGF0YSIsIHt9KSksCiAgICAgICAgbW9kZWw9TW9kZWxDb25maWcoKipyYXcuZ2V0KCJtb2RlbCIsIHt9KSksCiAgICAgICAgdHJhaW5pbmc9VHJhaW5pbmdDb25maWcoKipyYXcuZ2V0KCJ0cmFpbmluZyIsIHt9KSksCiAgICApCg==",
    "data.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFzdAppbXBvcnQganNvbgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHRvcmNoCmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IHRyYWluX3Rlc3Rfc3BsaXQKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhc2V0Cgpmcm9tIGFyZW5hX3Jhbmtlci5jb25maWcgaW1wb3J0IERhdGFDb25maWcsIE1vZGVsQ29uZmlnCgpMQUJFTF9DT0xVTU5TID0gWyJ3aW5uZXJfbW9kZWxfYSIsICJ3aW5uZXJfbW9kZWxfYiIsICJ3aW5uZXJfdGllIl0KTEFCRUxfVE9fSUQgPSB7Indpbm5lcl9tb2RlbF9hIjogMCwgIndpbm5lcl9tb2RlbF9iIjogMSwgIndpbm5lcl90aWUiOiAyfQpJRF9UT19MQUJFTCA9IHt2YWx1ZToga2V5IGZvciBrZXksIHZhbHVlIGluIExBQkVMX1RPX0lELml0ZW1zKCl9CgoKZGVmIF9wYXJzZV9jb252ZXJzYXRpb25fZmllbGQodmFsdWU6IEFueSkgLT4gbGlzdFtzdHJdOgogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgbGlzdCk6CiAgICAgICAgcmV0dXJuIFtzdHIoaXRlbSkgZm9yIGl0ZW0gaW4gdmFsdWVdCgogICAgaWYgdmFsdWUgaXMgTm9uZSBvciAoaXNpbnN0YW5jZSh2YWx1ZSwgZmxvYXQpIGFuZCBwZC5pc25hKHZhbHVlKSk6CiAgICAgICAgcmV0dXJuIFtdCgogICAgaWYgbm90IGlzaW5zdGFuY2UodmFsdWUsIHN0cik6CiAgICAgICAgcmV0dXJuIFtzdHIodmFsdWUpXQoKICAgIHRleHQgPSB2YWx1ZS5zdHJpcCgpCiAgICBpZiBub3QgdGV4dDoKICAgICAgICByZXR1cm4gW10KCiAgICBmb3IgcGFyc2VyIGluIChqc29uLmxvYWRzLCBhc3QubGl0ZXJhbF9ldmFsKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHBhcnNlZCA9IHBhcnNlcih0ZXh0KQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKHBhcnNlZCwgbGlzdCk6CiAgICAgICAgICAgICAgICByZXR1cm4gW3N0cihpdGVtKSBmb3IgaXRlbSBpbiBwYXJzZWRdCiAgICAgICAgZXhjZXB0IChWYWx1ZUVycm9yLCBTeW50YXhFcnJvcik6CiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgcmV0dXJuIFt0ZXh0XQoKCmRlZiBfbm9ybWFsaXplX3RleHQodmFsdWU6IEFueSwgbWF4X2NoYXJzOiBpbnQpIC0+IHN0cjoKICAgIGNodW5rcyA9IF9wYXJzZV9jb252ZXJzYXRpb25fZmllbGQodmFsdWUpCiAgICB0ZXh0ID0gIlxuIi5qb2luKGNodW5rLnN0cmlwKCkgZm9yIGNodW5rIGluIGNodW5rcyBpZiBzdHIoY2h1bmspLnN0cmlwKCkpCiAgICByZXR1cm4gdGV4dFs6bWF4X2NoYXJzXQoKCmRlZiBfYnVpbGRfdGFyZ2V0KHJvdzogcGQuU2VyaWVzKSAtPiBpbnQ6CiAgICBmb3IgY29sdW1uLCBsYWJlbF9pZCBpbiBMQUJFTF9UT19JRC5pdGVtcygpOgogICAgICAgIGlmIGludChyb3dbY29sdW1uXSkgPT0gMToKICAgICAgICAgICAgcmV0dXJuIGxhYmVsX2lkCiAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCB0YXJnZXQgcm93OiB7cm93W0xBQkVMX0NPTFVNTlNdLnRvX2RpY3QoKX0iKQoKCmRlZiBsb2FkX3RyYWluX2RhdGFmcmFtZShkYXRhX2Rpcjogc3RyIHwgUGF0aCwgY29uZmlnOiBEYXRhQ29uZmlnKSAtPiBwZC5EYXRhRnJhbWU6CiAgICBkZiA9IHBkLnJlYWRfY3N2KFBhdGgoZGF0YV9kaXIpIC8gY29uZmlnLnRyYWluX3BhdGgpCiAgICBkZiA9IGRmLmNvcHkoKQogICAgZGZbInByb21wdF90ZXh0Il0gPSBkZlsicHJvbXB0Il0ubWFwKGxhbWJkYSB2YWx1ZTogX25vcm1hbGl6ZV90ZXh0KHZhbHVlLCBjb25maWcudGV4dF9tYXhfY2hhcnMpKQogICAgZGZbInJlc3BvbnNlX2FfdGV4dCJdID0gZGZbInJlc3BvbnNlX2EiXS5tYXAobGFtYmRhIHZhbHVlOiBfbm9ybWFsaXplX3RleHQodmFsdWUsIGNvbmZpZy50ZXh0X21heF9jaGFycykpCiAgICBkZlsicmVzcG9uc2VfYl90ZXh0Il0gPSBkZlsicmVzcG9uc2VfYiJdLm1hcChsYW1iZGEgdmFsdWU6IF9ub3JtYWxpemVfdGV4dCh2YWx1ZSwgY29uZmlnLnRleHRfbWF4X2NoYXJzKSkKICAgIGRmWyJsYWJlbCJdID0gZGYuYXBwbHkoX2J1aWxkX3RhcmdldCwgYXhpcz0xKQogICAgcmV0dXJuIGRmCgoKZGVmIGxvYWRfdGVzdF9kYXRhZnJhbWUoZGF0YV9kaXI6IHN0ciB8IFBhdGgsIGNvbmZpZzogRGF0YUNvbmZpZykgLT4gcGQuRGF0YUZyYW1lOgogICAgZGYgPSBwZC5yZWFkX2NzdihQYXRoKGRhdGFfZGlyKSAvIGNvbmZpZy50ZXN0X3BhdGgpCiAgICBkZiA9IGRmLmNvcHkoKQogICAgZGZbInByb21wdF90ZXh0Il0gPSBkZlsicHJvbXB0Il0ubWFwKGxhbWJkYSB2YWx1ZTogX25vcm1hbGl6ZV90ZXh0KHZhbHVlLCBjb25maWcudGV4dF9tYXhfY2hhcnMpKQogICAgZGZbInJlc3BvbnNlX2FfdGV4dCJdID0gZGZbInJlc3BvbnNlX2EiXS5tYXAobGFtYmRhIHZhbHVlOiBfbm9ybWFsaXplX3RleHQodmFsdWUsIGNvbmZpZy50ZXh0X21heF9jaGFycykpCiAgICBkZlsicmVzcG9uc2VfYl90ZXh0Il0gPSBkZlsicmVzcG9uc2VfYiJdLm1hcChsYW1iZGEgdmFsdWU6IF9ub3JtYWxpemVfdGV4dCh2YWx1ZSwgY29uZmlnLnRleHRfbWF4X2NoYXJzKSkKICAgIHJldHVybiBkZgoKCmRlZiBzcGxpdF90cmFpbl92YWxpZChkZjogcGQuRGF0YUZyYW1lLCBjb25maWc6IERhdGFDb25maWcpIC0+IHR1cGxlW3BkLkRhdGFGcmFtZSwgcGQuRGF0YUZyYW1lXToKICAgIHRyYWluX2RmLCB2YWxpZF9kZiA9IHRyYWluX3Rlc3Rfc3BsaXQoCiAgICAgICAgZGYsCiAgICAgICAgdGVzdF9zaXplPWNvbmZpZy52YWxpZGF0aW9uX3NpemUsCiAgICAgICAgcmFuZG9tX3N0YXRlPWNvbmZpZy5yYW5kb21fc3RhdGUsCiAgICAgICAgc3RyYXRpZnk9ZGZbImxhYmVsIl0sCiAgICApCiAgICByZXR1cm4gdHJhaW5fZGYucmVzZXRfaW5kZXgoZHJvcD1UcnVlKSwgdmFsaWRfZGYucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQoKCmRlZiBfYmF0Y2hfZW5jb2RlKHRva2VuaXplciwgdGV4dHM6IGxpc3Rbc3RyXSwgbWF4X2xlbmd0aDogaW50KSAtPiBkaWN0W3N0ciwgdG9yY2guVGVuc29yXToKICAgIHJldHVybiB0b2tlbml6ZXIoCiAgICAgICAgdGV4dHMsCiAgICAgICAgcGFkZGluZz1UcnVlLAogICAgICAgIHRydW5jYXRpb249VHJ1ZSwKICAgICAgICBtYXhfbGVuZ3RoPW1heF9sZW5ndGgsCiAgICAgICAgcmV0dXJuX3RlbnNvcnM9InB0IiwKICAgICkKCgpAZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIEVuY29kZWRCYXRjaDoKICAgIHByb21wdDogZGljdFtzdHIsIHRvcmNoLlRlbnNvcl0KICAgIHJlc3BvbnNlX2E6IGRpY3Rbc3RyLCB0b3JjaC5UZW5zb3JdCiAgICByZXNwb25zZV9iOiBkaWN0W3N0ciwgdG9yY2guVGVuc29yXQogICAgbGFiZWxzOiB0b3JjaC5UZW5zb3IgfCBOb25lCiAgICBpZHM6IGxpc3RbaW50XQoKCmNsYXNzIEFyZW5hUHJlZmVyZW5jZURhdGFzZXQoRGF0YXNldCk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgZGF0YWZyYW1lOiBwZC5EYXRhRnJhbWUsIHdpdGhfbGFiZWxzOiBib29sKSAtPiBOb25lOgogICAgICAgIHNlbGYuZGF0YWZyYW1lID0gZGF0YWZyYW1lLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgICAgICBzZWxmLndpdGhfbGFiZWxzID0gd2l0aF9sYWJlbHMKCiAgICBkZWYgX19sZW5fXyhzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIGxlbihzZWxmLmRhdGFmcmFtZSkKCiAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaW5kZXg6IGludCkgLT4gZGljdFtzdHIsIEFueV06CiAgICAgICAgcm93ID0gc2VsZi5kYXRhZnJhbWUuaWxvY1tpbmRleF0KICAgICAgICBpdGVtID0gewogICAgICAgICAgICAiaWQiOiBpbnQocm93WyJpZCJdKSwKICAgICAgICAgICAgInByb21wdF90ZXh0Ijogcm93WyJwcm9tcHRfdGV4dCJdLAogICAgICAgICAgICAicmVzcG9uc2VfYV90ZXh0Ijogcm93WyJyZXNwb25zZV9hX3RleHQiXSwKICAgICAgICAgICAgInJlc3BvbnNlX2JfdGV4dCI6IHJvd1sicmVzcG9uc2VfYl90ZXh0Il0sCiAgICAgICAgfQogICAgICAgIGlmIHNlbGYud2l0aF9sYWJlbHM6CiAgICAgICAgICAgIGl0ZW1bImxhYmVsIl0gPSBpbnQocm93WyJsYWJlbCJdKQogICAgICAgIHJldHVybiBpdGVtCgoKY2xhc3MgQXJlbmFDb2xsYXRvcjoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCB0b2tlbml6ZXIsIG1vZGVsX2NvbmZpZzogTW9kZWxDb25maWcpIC0+IE5vbmU6CiAgICAgICAgc2VsZi50b2tlbml6ZXIgPSB0b2tlbml6ZXIKICAgICAgICBzZWxmLm1vZGVsX2NvbmZpZyA9IG1vZGVsX2NvbmZpZwoKICAgIGRlZiBfX2NhbGxfXyhzZWxmLCBiYXRjaDogbGlzdFtkaWN0W3N0ciwgQW55XV0pIC0+IEVuY29kZWRCYXRjaDoKICAgICAgICBwcm9tcHQgPSBfYmF0Y2hfZW5jb2RlKHNlbGYudG9rZW5pemVyLCBbaXRlbVsicHJvbXB0X3RleHQiXSBmb3IgaXRlbSBpbiBiYXRjaF0sIHNlbGYubW9kZWxfY29uZmlnLm1heF9sZW5ndGgpCiAgICAgICAgcmVzcG9uc2VfYSA9IF9iYXRjaF9lbmNvZGUoc2VsZi50b2tlbml6ZXIsIFtpdGVtWyJyZXNwb25zZV9hX3RleHQiXSBmb3IgaXRlbSBpbiBiYXRjaF0sIHNlbGYubW9kZWxfY29uZmlnLm1heF9sZW5ndGgpCiAgICAgICAgcmVzcG9uc2VfYiA9IF9iYXRjaF9lbmNvZGUoc2VsZi50b2tlbml6ZXIsIFtpdGVtWyJyZXNwb25zZV9iX3RleHQiXSBmb3IgaXRlbSBpbiBiYXRjaF0sIHNlbGYubW9kZWxfY29uZmlnLm1heF9sZW5ndGgpCiAgICAgICAgbGFiZWxzID0gTm9uZQogICAgICAgIGlmICJsYWJlbCIgaW4gYmF0Y2hbMF06CiAgICAgICAgICAgIGxhYmVscyA9IHRvcmNoLnRlbnNvcihbaXRlbVsibGFiZWwiXSBmb3IgaXRlbSBpbiBiYXRjaF0sIGR0eXBlPXRvcmNoLmxvbmcpCiAgICAgICAgcmV0dXJuIEVuY29kZWRCYXRjaCgKICAgICAgICAgICAgcHJvbXB0PXByb21wdCwKICAgICAgICAgICAgcmVzcG9uc2VfYT1yZXNwb25zZV9hLAogICAgICAgICAgICByZXNwb25zZV9iPXJlc3BvbnNlX2IsCiAgICAgICAgICAgIGxhYmVscz1sYWJlbHMsCiAgICAgICAgICAgIGlkcz1baXRlbVsiaWQiXSBmb3IgaXRlbSBpbiBiYXRjaF0sCiAgICAgICAgKQo=",
    "hf.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgdHJhbnNmb3JtZXJzCmZyb20gcGVmdCBpbXBvcnQgTG9yYUNvbmZpZywgVGFza1R5cGUsIGdldF9wZWZ0X21vZGVsCmZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWwsIEF1dG9Ub2tlbml6ZXIKCmZyb20gYXJlbmFfcmFua2VyLmNvbmZpZyBpbXBvcnQgTW9kZWxDb25maWcKCgpkZWYgX2lzX3RyYW5zZm9ybWVyc192NV9vcl9uZXdlcigpIC0+IGJvb2w6CiAgICBtYWpvciwgKl8gPSB0cmFuc2Zvcm1lcnMuX192ZXJzaW9uX18uc3BsaXQoIi4iLCAxKQogICAgcmV0dXJuIGludChtYWpvcikgPj0gNQoKCmRlZiBlbnN1cmVfc3VwcG9ydGVkX3RyYW5zZm9ybWVyc192ZXJzaW9uKCkgLT4gTm9uZToKICAgIGlmIG5vdCBfaXNfdHJhbnNmb3JtZXJzX3Y1X29yX25ld2VyKCk6CiAgICAgICAgcmV0dXJuCgogICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICLlvZPliY3njq/looPnmoQgdHJhbnNmb3JtZXJzIOeJiOacrOS4uiAiCiAgICAgICAgZiJ7dHJhbnNmb3JtZXJzLl9fdmVyc2lvbl9ffeOAguivpemhueebruWPqumqjOivgeS6hiA0Lngg54mI5pys77yM5bm25LiU5bey5ZyoIHB5cHJvamVjdC50b21sIOS4reaUtue0p+S4uiAiCiAgICAgICAgIlwidHJhbnNmb3JtZXJzPj00LjU1LjAsPDVcIuOAgiIKICAgICAgICAi6K+35YWI5Zyo6aG555uu55uu5b2V5omn6KGMIGB1diBzeW5jYO+8jOWGjemHjeaWsOi/kOihjOiuree7g+aIlumihOa1i+OAgiIKICAgICkKCgpkZWYgX2Rlc2NyaWJlX21vZGVsX3NvdXJjZShtb2RlbF9uYW1lOiBzdHIpIC0+IHN0cjoKICAgIHNvdXJjZV9wYXRoID0gUGF0aChtb2RlbF9uYW1lKS5leHBhbmR1c2VyKCkKICAgIGlmIHNvdXJjZV9wYXRoLmV4aXN0cygpOgogICAgICAgIGNvbmZpZ19wYXRoID0gc291cmNlX3BhdGggLyAiY29uZmlnLmpzb24iCiAgICAgICAgaWYgY29uZmlnX3BhdGguZXhpc3RzKCk6CiAgICAgICAgICAgIHJldHVybiBmIuajgOa1i+WIsOacrOWcsOaooeWei+ebruW9le+8mntzb3VyY2VfcGF0aH0iCiAgICAgICAgcmV0dXJuICgKICAgICAgICAgICAgZiLmo4DmtYvliLDlkIzlkI3mnKzlnLDnm67lvZXvvJp7c291cmNlX3BhdGh977yM5L2G5YW25Lit57y65bCRIGBjb25maWcuanNvbmDjgIIiCiAgICAgICAgICAgICLov5nkvJrorqkgdHJhbnNmb3JtZXJzIOaKiuWug+W9k+aIkOacrOWcsOaooeWei+ebruW9leWkhOeQhu+8jOW5tuebtOaOpeWKoOi9veWksei0peOAgiIKICAgICAgICApCgogICAgaWYgIi8iIGluIG1vZGVsX25hbWU6CiAgICAgICAgcmV0dXJuICgKICAgICAgICAgICAgZiJge21vZGVsX25hbWV9YCDkvJrooqvlvZPkvZwgSHVnZ2luZyBGYWNlIOS7k+W6kyBJROOAgiIKICAgICAgICAgICAgIummluasoei/kOihjOmcgOimgeiBlOe9keS4i+i9ve+8m+emu+e6v+eOr+Wig+S4i+ivt+WFiOmihOS4i+i9veWIsOacrOWcsOebruW9le+8jOWGjeaKiiBgbW9kZWxfbmFtZWAg5pS55oiQ5pys5Zyw57ud5a+56Lev5b6E44CCIgogICAgICAgICkKCiAgICByZXR1cm4gZiJge21vZGVsX25hbWV9YCDml6LkuI3mmK/lt7LlrZjlnKjnmoTmnKzlnLDnm67lvZXvvIzkuZ/kuI3mmK/moIflh4bnmoQgSHVnZ2luZyBGYWNlIOS7k+W6kyBJROOAgiIKCgpkZWYgX2xvYWRfZXJyb3IoYWN0aW9uOiBzdHIsIG1vZGVsX25hbWU6IHN0ciwgZXhjOiBPU0Vycm9yKSAtPiBSdW50aW1lRXJyb3I6CiAgICByZXR1cm4gUnVudGltZUVycm9yKAogICAgICAgIGYie2FjdGlvbn3lpLHotKXvvJp7ZXhjfVxuIgogICAgICAgIGYie19kZXNjcmliZV9tb2RlbF9zb3VyY2UobW9kZWxfbmFtZSl9XG4iCiAgICAgICAgIuaOkuafpeW7uuiuru+8mlxuIgogICAgICAgICIxLiDlpoLmnpzkvaDkvp3otZblnKjnur/kuIvovb3vvIznoa7orqTlvZPliY3njq/looPog73orr/pl64gaHVnZ2luZ2ZhY2UuY2/jgIJcbiIKICAgICAgICAiMi4g5aaC5p6c5L2g5Zyo56a757q/546v5aKD6L+Q6KGM77yM5YWI5oqK5qih5Z6L5LiL6L295Yiw5pys5Zyw77yM5YaN5oqKIGBtb2RlbF9uYW1lYCDmlLnkuLrmnKzlnLDot6/lvoTjgIJcbiIKICAgICAgICAiMy4g5aaC5p6c5bel5L2c55uu5b2V5LiL5a2Y5Zyo5ZCM5ZCN55uu5b2VIGBRd2VuL1F3ZW4zLUVtYmVkZGluZy0wLjZCYO+8jOivt+WIoOmZpOaIlumHjeWRveWQjeivpeebruW9leOAgiIKICAgICkKCgpkZWYgbG9hZF90b2tlbml6ZXIoY29uZmlnOiBNb2RlbENvbmZpZyk6CiAgICBlbnN1cmVfc3VwcG9ydGVkX3RyYW5zZm9ybWVyc192ZXJzaW9uKCkKICAgIHRyeToKICAgICAgICByZXR1cm4gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgICAgIGNvbmZpZy5tb2RlbF9uYW1lLAogICAgICAgICAgICB0cnVzdF9yZW1vdGVfY29kZT1UcnVlLAogICAgICAgICAgICBjYWNoZV9kaXI9Y29uZmlnLmNhY2hlX2RpciwKICAgICAgICAgICAgbG9jYWxfZmlsZXNfb25seT1jb25maWcubG9jYWxfZmlsZXNfb25seSwKICAgICAgICApCiAgICBleGNlcHQgT1NFcnJvciBhcyBleGM6CiAgICAgICAgcmFpc2UgX2xvYWRfZXJyb3IoIuWKoOi9vSB0b2tlbml6ZXIiLCBjb25maWcubW9kZWxfbmFtZSwgZXhjKQoKCmRlZiBsb2FkX2VuY29kZXIoY29uZmlnOiBNb2RlbENvbmZpZyk6CiAgICBlbnN1cmVfc3VwcG9ydGVkX3RyYW5zZm9ybWVyc192ZXJzaW9uKCkKICAgIHRyeToKICAgICAgICBlbmNvZGVyID0gQXV0b01vZGVsLmZyb21fcHJldHJhaW5lZCgKICAgICAgICAgICAgY29uZmlnLm1vZGVsX25hbWUsCiAgICAgICAgICAgIHRydXN0X3JlbW90ZV9jb2RlPVRydWUsCiAgICAgICAgICAgIGNhY2hlX2Rpcj1jb25maWcuY2FjaGVfZGlyLAogICAgICAgICAgICBsb2NhbF9maWxlc19vbmx5PWNvbmZpZy5sb2NhbF9maWxlc19vbmx5LAogICAgICAgICkKICAgICAgICBpZiBub3QgY29uZmlnLnVzZV9sb3JhOgogICAgICAgICAgICByZXR1cm4gZW5jb2RlcgoKICAgICAgICByZXR1cm4gZ2V0X3BlZnRfbW9kZWwoCiAgICAgICAgICAgIGVuY29kZXIsCiAgICAgICAgICAgIExvcmFDb25maWcoCiAgICAgICAgICAgICAgICByPWNvbmZpZy5sb3JhX3IsCiAgICAgICAgICAgICAgICBsb3JhX2FscGhhPWNvbmZpZy5sb3JhX2FscGhhLAogICAgICAgICAgICAgICAgbG9yYV9kcm9wb3V0PWNvbmZpZy5sb3JhX2Ryb3BvdXQsCiAgICAgICAgICAgICAgICBiaWFzPWNvbmZpZy5sb3JhX2JpYXMsCiAgICAgICAgICAgICAgICB0YXJnZXRfbW9kdWxlcz1jb25maWcubG9yYV90YXJnZXRfbW9kdWxlcywKICAgICAgICAgICAgICAgIHRhc2tfdHlwZT1UYXNrVHlwZVtjb25maWcubG9yYV90YXNrX3R5cGUudXBwZXIoKV0sCiAgICAgICAgICAgICksCiAgICAgICAgKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgIHJhaXNlIF9sb2FkX2Vycm9yKCLliqDovb0gZW5jb2RlciIsIGNvbmZpZy5tb2RlbF9uYW1lLCBleGMpCiAgICBleGNlcHQgKEtleUVycm9yLCBWYWx1ZUVycm9yKSBhcyBleGM6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYi5peg5pWI55qEIExvUkEgdGFza190eXBlIOmFjee9ru+8mntjb25maWcubG9yYV90YXNrX3R5cGV9IikgZnJvbSBleGMK",
    "modeling.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgbm4KCmZyb20gYXJlbmFfcmFua2VyLmNvbmZpZyBpbXBvcnQgTW9kZWxDb25maWcKZnJvbSBhcmVuYV9yYW5rZXIuaGYgaW1wb3J0IGxvYWRfZW5jb2RlcgoKCmRlZiBtYXNrZWRfbWVhbl9wb29sKGxhc3RfaGlkZGVuX3N0YXRlOiB0b3JjaC5UZW5zb3IsIGF0dGVudGlvbl9tYXNrOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgIG1hc2sgPSBhdHRlbnRpb25fbWFzay51bnNxdWVlemUoLTEpLnRvKGxhc3RfaGlkZGVuX3N0YXRlLmR0eXBlKQogICAgbWFza2VkID0gbGFzdF9oaWRkZW5fc3RhdGUgKiBtYXNrCiAgICBkZW5vbSA9IG1hc2suc3VtKGRpbT0xKS5jbGFtcChtaW49MWUtNikKICAgIHJldHVybiBtYXNrZWQuc3VtKGRpbT0xKSAvIGRlbm9tCgoKQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBNb2RlbE91dHB1dDoKICAgIGxvZ2l0czogdG9yY2guVGVuc29yCiAgICBsb3NzOiB0b3JjaC5UZW5zb3IgfCBOb25lID0gTm9uZQoKCmNsYXNzIFByZWZlcmVuY2VDbGFzc2lmaWVyKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgY29uZmlnOiBNb2RlbENvbmZpZykgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbmZpZyA9IGNvbmZpZwogICAgICAgIHNlbGYuZW5jb2RlciA9IGxvYWRfZW5jb2Rlcihjb25maWcpCiAgICAgICAgaGlkZGVuX3NpemUgPSBzZWxmLmVuY29kZXIuY29uZmlnLmhpZGRlbl9zaXplCiAgICAgICAgY2xhc3NpZmllcl9pbnB1dCA9IGhpZGRlbl9zaXplICogNgogICAgICAgIHNlbGYuZHJvcG91dCA9IG5uLkRyb3BvdXQoY29uZmlnLmRyb3BvdXQpCiAgICAgICAgc2VsZi5jbGFzc2lmaWVyID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uTGluZWFyKGNsYXNzaWZpZXJfaW5wdXQsIGhpZGRlbl9zaXplICogMiksCiAgICAgICAgICAgIG5uLkdFTFUoKSwKICAgICAgICAgICAgbm4uRHJvcG91dChjb25maWcuZHJvcG91dCksCiAgICAgICAgICAgIG5uLkxpbmVhcihoaWRkZW5fc2l6ZSAqIDIsIDMpLAogICAgICAgICkKICAgICAgICBzZWxmLmxvc3NfZm4gPSBubi5Dcm9zc0VudHJvcHlMb3NzKCkKCiAgICAgICAgaWYgY29uZmlnLmZyZWV6ZV9lbmNvZGVyOgogICAgICAgICAgICBmb3IgcGFyYW1ldGVyIGluIHNlbGYuZW5jb2Rlci5wYXJhbWV0ZXJzKCk6CiAgICAgICAgICAgICAgICBwYXJhbWV0ZXIucmVxdWlyZXNfZ3JhZCA9IEZhbHNlCgogICAgZGVmIHByaW50X3RyYWluYWJsZV9wYXJhbWV0ZXJzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgdG90YWxfcGFyYW1zID0gMAogICAgICAgIHRyYWluYWJsZV9wYXJhbXMgPSAwCiAgICAgICAgZm9yIHBhcmFtZXRlciBpbiBzZWxmLnBhcmFtZXRlcnMoKToKICAgICAgICAgICAgY291bnQgPSBwYXJhbWV0ZXIubnVtZWwoKQogICAgICAgICAgICB0b3RhbF9wYXJhbXMgKz0gY291bnQKICAgICAgICAgICAgaWYgcGFyYW1ldGVyLnJlcXVpcmVzX2dyYWQ6CiAgICAgICAgICAgICAgICB0cmFpbmFibGVfcGFyYW1zICs9IGNvdW50CgogICAgICAgIHJhdGlvID0gMC4wIGlmIHRvdGFsX3BhcmFtcyA9PSAwIGVsc2UgdHJhaW5hYmxlX3BhcmFtcyAvIHRvdGFsX3BhcmFtcyAqIDEwMAogICAgICAgIHByaW50KAogICAgICAgICAgICBmInRyYWluYWJsZSBwYXJhbXM6IHt0cmFpbmFibGVfcGFyYW1zfSB8fCBhbGwgcGFyYW1zOiB7dG90YWxfcGFyYW1zfSB8fCAiCiAgICAgICAgICAgIGYidHJhaW5hYmxlJToge3JhdGlvOi40Zn0iCiAgICAgICAgKQoKICAgIGRlZiBlbmFibGVfZ3JhZGllbnRfY2hlY2twb2ludGluZyhzZWxmKSAtPiBOb25lOgogICAgICAgIGlmIGhhc2F0dHIoc2VsZi5lbmNvZGVyLCAiZ3JhZGllbnRfY2hlY2twb2ludGluZ19lbmFibGUiKToKICAgICAgICAgICAgc2VsZi5lbmNvZGVyLmdyYWRpZW50X2NoZWNrcG9pbnRpbmdfZW5hYmxlKCkKICAgICAgICBlbGlmIGhhc2F0dHIoc2VsZi5lbmNvZGVyLCAiYmFzZV9tb2RlbCIpIGFuZCBoYXNhdHRyKHNlbGYuZW5jb2Rlci5iYXNlX21vZGVsLCAiZ3JhZGllbnRfY2hlY2twb2ludGluZ19lbmFibGUiKToKICAgICAgICAgICAgc2VsZi5lbmNvZGVyLmJhc2VfbW9kZWwuZ3JhZGllbnRfY2hlY2twb2ludGluZ19lbmFibGUoKQoKICAgICAgICBpZiBoYXNhdHRyKHNlbGYuZW5jb2RlciwgImVuYWJsZV9pbnB1dF9yZXF1aXJlX2dyYWRzIik6CiAgICAgICAgICAgIHNlbGYuZW5jb2Rlci5lbmFibGVfaW5wdXRfcmVxdWlyZV9ncmFkcygpCiAgICAgICAgZWxpZiBoYXNhdHRyKHNlbGYuZW5jb2RlciwgImJhc2VfbW9kZWwiKSBhbmQgaGFzYXR0cihzZWxmLmVuY29kZXIuYmFzZV9tb2RlbCwgImVuYWJsZV9pbnB1dF9yZXF1aXJlX2dyYWRzIik6CiAgICAgICAgICAgIHNlbGYuZW5jb2Rlci5iYXNlX21vZGVsLmVuYWJsZV9pbnB1dF9yZXF1aXJlX2dyYWRzKCkKCiAgICAgICAgZW5jb2Rlcl9jb25maWcgPSBnZXRhdHRyKHNlbGYuZW5jb2RlciwgImNvbmZpZyIsIE5vbmUpCiAgICAgICAgaWYgZW5jb2Rlcl9jb25maWcgaXMgbm90IE5vbmUgYW5kIGhhc2F0dHIoZW5jb2Rlcl9jb25maWcsICJ1c2VfY2FjaGUiKToKICAgICAgICAgICAgZW5jb2Rlcl9jb25maWcudXNlX2NhY2hlID0gRmFsc2UKCiAgICAgICAgYmFzZV9tb2RlbCA9IGdldGF0dHIoc2VsZi5lbmNvZGVyLCAiYmFzZV9tb2RlbCIsIE5vbmUpCiAgICAgICAgYmFzZV9jb25maWcgPSBnZXRhdHRyKGJhc2VfbW9kZWwsICJjb25maWciLCBOb25lKQogICAgICAgIGlmIGJhc2VfY29uZmlnIGlzIG5vdCBOb25lIGFuZCBoYXNhdHRyKGJhc2VfY29uZmlnLCAidXNlX2NhY2hlIik6CiAgICAgICAgICAgIGJhc2VfY29uZmlnLnVzZV9jYWNoZSA9IEZhbHNlCgogICAgZGVmIGVuY29kZShzZWxmLCBpbnB1dHM6IGRpY3Rbc3RyLCB0b3JjaC5UZW5zb3JdKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgb3V0cHV0cyA9IHNlbGYuZW5jb2RlcigqKmlucHV0cykKICAgICAgICByZXR1cm4gbWFza2VkX21lYW5fcG9vbChvdXRwdXRzLmxhc3RfaGlkZGVuX3N0YXRlLCBpbnB1dHNbImF0dGVudGlvbl9tYXNrIl0pCgogICAgZGVmIGZvcndhcmQoCiAgICAgICAgc2VsZiwKICAgICAgICBwcm9tcHRfaW5wdXRzOiBkaWN0W3N0ciwgdG9yY2guVGVuc29yXSwKICAgICAgICByZXNwb25zZV9hX2lucHV0czogZGljdFtzdHIsIHRvcmNoLlRlbnNvcl0sCiAgICAgICAgcmVzcG9uc2VfYl9pbnB1dHM6IGRpY3Rbc3RyLCB0b3JjaC5UZW5zb3JdLAogICAgICAgIGxhYmVsczogdG9yY2guVGVuc29yIHwgTm9uZSA9IE5vbmUsCiAgICApIC0+IE1vZGVsT3V0cHV0OgogICAgICAgIHByb21wdF9lbWIgPSBzZWxmLmVuY29kZShwcm9tcHRfaW5wdXRzKQogICAgICAgIHJlc3BvbnNlX2FfZW1iID0gc2VsZi5lbmNvZGUocmVzcG9uc2VfYV9pbnB1dHMpCiAgICAgICAgcmVzcG9uc2VfYl9lbWIgPSBzZWxmLmVuY29kZShyZXNwb25zZV9iX2lucHV0cykKCiAgICAgICAgZmVhdHVyZXMgPSB0b3JjaC5jYXQoCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICAgIHByb21wdF9lbWIsCiAgICAgICAgICAgICAgICByZXNwb25zZV9hX2VtYiwKICAgICAgICAgICAgICAgIHJlc3BvbnNlX2JfZW1iLAogICAgICAgICAgICAgICAgcmVzcG9uc2VfYV9lbWIgLSByZXNwb25zZV9iX2VtYiwKICAgICAgICAgICAgICAgIHJlc3BvbnNlX2FfZW1iIC0gcHJvbXB0X2VtYiwKICAgICAgICAgICAgICAgIHJlc3BvbnNlX2JfZW1iIC0gcHJvbXB0X2VtYiwKICAgICAgICAgICAgXSwKICAgICAgICAgICAgZGltPS0xLAogICAgICAgICkKICAgICAgICBsb2dpdHMgPSBzZWxmLmNsYXNzaWZpZXIoc2VsZi5kcm9wb3V0KGZlYXR1cmVzKSkKCiAgICAgICAgbG9zcyA9IHNlbGYubG9zc19mbihsb2dpdHMsIGxhYmVscykgaWYgbGFiZWxzIGlzIG5vdCBOb25lIGVsc2UgTm9uZQogICAgICAgIHJldHVybiBNb2RlbE91dHB1dChsb2dpdHM9bG9naXRzLCBsb3NzPWxvc3MpCg==",
    "train.py": "...",
    "predict.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBsb2dnaW5nCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyCmZyb20gdHFkbS5hdXRvIGltcG9ydCB0cWRtCgpmcm9tIGFyZW5hX3Jhbmtlci5jb25maWcgaW1wb3J0IGxvYWRfY29uZmlnCmZyb20gYXJlbmFfcmFua2VyLmRhdGEgaW1wb3J0IEFyZW5hQ29sbGF0b3IsIEFyZW5hUHJlZmVyZW5jZURhdGFzZXQsIElEX1RPX0xBQkVMLCBsb2FkX3Rlc3RfZGF0YWZyYW1lCmZyb20gYXJlbmFfcmFua2VyLmhmIGltcG9ydCBlbnN1cmVfc3VwcG9ydGVkX3RyYW5zZm9ybWVyc192ZXJzaW9uCmZyb20gYXJlbmFfcmFua2VyLm1vZGVsaW5nIGltcG9ydCBQcmVmZXJlbmNlQ2xhc3NpZmllcgoKCkxPR0dFUiA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJhcmVuYV9yYW5rZXIucHJlZGljdCIpClBST0JBQklMSVRZX0VQU0lMT04gPSAxZS02CgoKZGVmIHBhcnNlX2FyZ3MoKSAtPiBhcmdwYXJzZS5OYW1lc3BhY2U6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iR2VuZXJhdGUgc3VibWlzc2lvbiB3aXRoIHRyYWluZWQgQXJlbmEgcmFua2VyLiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNoZWNrcG9pbnQtZGlyIiwgdHlwZT1zdHIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGEtZGlyIiwgdHlwZT1zdHIsIGRlZmF1bHQ9Ii4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQtcGF0aCIsIHR5cGU9c3RyLCBkZWZhdWx0PU5vbmUpCiAgICByZXR1cm4gcGFyc2VyLnBhcnNlX2FyZ3MoKQoKCmRlZiBtb3ZlX2lucHV0c190b19kZXZpY2UoaW5wdXRzOiBkaWN0W3N0ciwgdG9yY2guVGVuc29yXSwgZGV2aWNlOiB0b3JjaC5kZXZpY2UpIC0+IGRpY3Rbc3RyLCB0b3JjaC5UZW5zb3JdOgogICAgcmV0dXJuIHtrZXk6IHZhbHVlLnRvKGRldmljZSkgZm9yIGtleSwgdmFsdWUgaW4gaW5wdXRzLml0ZW1zKCl9CgoKZGVmIHNldHVwX2xvZ2dpbmcoKSAtPiBOb25lOgogICAgbG9nZ2luZy5iYXNpY0NvbmZpZygKICAgICAgICBsZXZlbD1sb2dnaW5nLklORk8sCiAgICAgICAgZm9ybWF0PSIlKGFzY3RpbWUpcyB8ICUobGV2ZWxuYW1lKXMgfCAlKG1lc3NhZ2UpcyIsCiAgICAgICAgZGF0ZWZtdD0iJUg6JU06JVMiLAogICAgKQoKCmRlZiBmb3JtYXRfc2Vjb25kcyhzZWNvbmRzOiBmbG9hdCkgLT4gc3RyOgogICAgdG90YWxfc2Vjb25kcyA9IG1heChpbnQoc2Vjb25kcyksIDApCiAgICBtaW51dGVzLCBzZWNzID0gZGl2bW9kKHRvdGFsX3NlY29uZHMsIDYwKQogICAgaG91cnMsIG1pbnV0ZXMgPSBkaXZtb2QobWludXRlcywgNjApCiAgICBpZiBob3VycyA+IDA6CiAgICAgICAgcmV0dXJuIGYie2hvdXJzfWgge21pbnV0ZXN9bSB7c2Vjc31zIgogICAgaWYgbWludXRlcyA+IDA6CiAgICAgICAgcmV0dXJuIGYie21pbnV0ZXN9bSB7c2Vjc31zIgogICAgcmV0dXJuIGYie3NlY3N9cyIKCgpkZWYgZGVzY3JpYmVfZGV2aWNlKGRldmljZTogdG9yY2guZGV2aWNlKSAtPiBzdHI6CiAgICBpZiBkZXZpY2UudHlwZSAhPSAiY3VkYSI6CiAgICAgICAgcmV0dXJuICJDUFUiCiAgICBncHVfbmFtZSA9IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKGRldmljZSkKICAgIHRvdGFsX21lbW9yeV9nYiA9IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9wcm9wZXJ0aWVzKGRldmljZSkudG90YWxfbWVtb3J5IC8gMTAyNCoqMwogICAgcmV0dXJuIGYie2dwdV9uYW1lfSAoe3RvdGFsX21lbW9yeV9nYjouMWZ9IEdCKSIKCgpkZWYgbm9ybWFsaXplX3Byb2JhYmlsaXRpZXMobG9naXRzOiB0b3JjaC5UZW5zb3IsIGVwc2lsb246IGZsb2F0ID0gUFJPQkFCSUxJVFlfRVBTSUxPTikgLT4gdG9yY2guVGVuc29yOgogICAgcHJvYnMgPSB0b3JjaC5zb2Z0bWF4KGxvZ2l0cywgZGltPS0xKQogICAgcHJvYnMgPSBwcm9icy5jbGFtcChtaW49ZXBzaWxvbiwgbWF4PTEuMCAtIGVwc2lsb24pCiAgICByZXR1cm4gcHJvYnMgLyBwcm9icy5zdW0oZGltPS0xLCBrZWVwZGltPVRydWUpCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgc2V0dXBfbG9nZ2luZygpCiAgICBhcmdzID0gcGFyc2VfYXJncygpCiAgICBjaGVja3BvaW50X2RpciA9IFBhdGgoYXJncy5jaGVja3BvaW50X2RpcikKICAgIGNvbmZpZyA9IGxvYWRfY29uZmlnKGNoZWNrcG9pbnRfZGlyIC8gImNvbmZpZy55YW1sIikKICAgIHN0YXJ0ZWRfYXQgPSB0aW1lLnBlcmZfY291bnRlcigpCgogICAgZW5zdXJlX3N1cHBvcnRlZF90cmFuc2Zvcm1lcnNfdmVyc2lvbigpCiAgICB0b2tlbml6ZXJfZGlyID0gY2hlY2twb2ludF9kaXIgLyAidG9rZW5pemVyIgogICAgaWYgbm90ICh0b2tlbml6ZXJfZGlyIC8gInRva2VuaXplcl9jb25maWcuanNvbiIpLmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIue8uuWwkSB0b2tlbml6ZXIg55uu5b2V5oiW5paH5Lu277yae3Rva2VuaXplcl9kaXJ9IikKICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvVG9rZW5pemVyCgogICAgdG9rZW5pemVyID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQodG9rZW5pemVyX2RpciwgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSkKICAgIG1vZGVsID0gUHJlZmVyZW5jZUNsYXNzaWZpZXIoY29uZmlnLm1vZGVsKQogICAgc3RhdGVfZGljdCA9IHRvcmNoLmxvYWQoY2hlY2twb2ludF9kaXIgLyAibW9kZWwucHQiLCBtYXBfbG9jYXRpb249ImNwdSIpCiAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3Qoc3RhdGVfZGljdCkKCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIG1vZGVsLnRvKGRldmljZSkKICAgIG1vZGVsLmV2YWwoKQogICAgTE9HR0VSLmluZm8oIumihOa1i+WQr+WKqCIpCiAgICBMT0dHRVIuaW5mbygiY2hlY2twb2ludDogJXMiLCBjaGVja3BvaW50X2RpcikKICAgIExPR0dFUi5pbmZvKCLorr7lpIc6ICVzIiwgZGVzY3JpYmVfZGV2aWNlKGRldmljZSkpCgogICAgdGVzdF9kZiA9IGxvYWRfdGVzdF9kYXRhZnJhbWUoYXJncy5kYXRhX2RpciwgY29uZmlnLmRhdGEpCiAgICBsb2FkZXIgPSBEYXRhTG9hZGVyKAogICAgICAgIEFyZW5hUHJlZmVyZW5jZURhdGFzZXQodGVzdF9kZiwgd2l0aF9sYWJlbHM9RmFsc2UpLAogICAgICAgIGJhdGNoX3NpemU9Y29uZmlnLnRyYWluaW5nLmJhdGNoX3NpemUsCiAgICAgICAgc2h1ZmZsZT1GYWxzZSwKICAgICAgICBudW1fd29ya2Vycz1jb25maWcudHJhaW5pbmcubnVtX3dvcmtlcnMsCiAgICAgICAgY29sbGF0ZV9mbj1BcmVuYUNvbGxhdG9yKHRva2VuaXplciwgY29uZmlnLm1vZGVsKSwKICAgICkKICAgIExPR0dFUi5pbmZvKAogICAgICAgICLlvoXpooTmtYvmoLfmnKw6ICVzIHwgYmF0Y2hfc2l6ZT0lcyB8IG1heF9sZW5ndGg9JXMiLAogICAgICAgIGxlbih0ZXN0X2RmKSwKICAgICAgICBjb25maWcudHJhaW5pbmcuYmF0Y2hfc2l6ZSwKICAgICAgICBjb25maWcubW9kZWwubWF4X2xlbmd0aCwKICAgICkKCiAgICByb3dzID0gW10KICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIGZvciBiYXRjaCBpbiB0cWRtKGxvYWRlciwgZGVzYz0icHJlZGljdCIpOgogICAgICAgICAgICBvdXRwdXRzID0gbW9kZWwoCiAgICAgICAgICAgICAgICBwcm9tcHRfaW5wdXRzPW1vdmVfaW5wdXRzX3RvX2RldmljZShiYXRjaC5wcm9tcHQsIGRldmljZSksCiAgICAgICAgICAgICAgICByZXNwb25zZV9hX2lucHV0cz1tb3ZlX2lucHV0c190b19kZXZpY2UoYmF0Y2gucmVzcG9uc2VfYSwgZGV2aWNlKSwKICAgICAgICAgICAgICAgIHJlc3BvbnNlX2JfaW5wdXRzPW1vdmVfaW5wdXRzX3RvX2RldmljZShiYXRjaC5yZXNwb25zZV9iLCBkZXZpY2UpLAogICAgICAgICAgICApCiAgICAgICAgICAgIHByb2JzID0gbm9ybWFsaXplX3Byb2JhYmlsaXRpZXMob3V0cHV0cy5sb2dpdHMpLmNwdSgpLm51bXB5KCkKICAgICAgICAgICAgZm9yIHNhbXBsZV9pZCwgcHJvYiBpbiB6aXAoYmF0Y2guaWRzLCBwcm9icywgc3RyaWN0PVRydWUpOgogICAgICAgICAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICAgICAiaWQiOiBzYW1wbGVfaWQsCiAgICAgICAgICAgICAgICAgICAgICAgIElEX1RPX0xBQkVMWzBdOiBwcm9iWzBdLAogICAgICAgICAgICAgICAgICAgICAgICBJRF9UT19MQUJFTFsxXTogcHJvYlsxXSwKICAgICAgICAgICAgICAgICAgICAgICAgSURfVE9fTEFCRUxbMl06IHByb2JbMl0sCiAgICAgICAgICAgICAgICAgICAgfQogICAgICAgICAgICAgICAgKQoKICAgIG91dHB1dF9wYXRoID0gUGF0aChhcmdzLm91dHB1dF9wYXRoKSBpZiBhcmdzLm91dHB1dF9wYXRoIGVsc2UgY2hlY2twb2ludF9kaXIgLyAic3VibWlzc2lvbi5jc3YiCiAgICBwZC5EYXRhRnJhbWUocm93cykudG9fY3N2KG91dHB1dF9wYXRoLCBpbmRleD1GYWxzZSkKICAgIExPR0dFUi5pbmZvKCLpooTmtYvlrozmiJDvvIznlJ/miJAgJXMg6KGM57uT5p6c77yM6ICX5pe2PSVzIiwgbGVuKHJvd3MpLCBmb3JtYXRfc2Vjb25kcyh0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnRlZF9hdCkpCiAgICBMT0dHRVIuaW5mbygic3VibWlzc2lvbiDlt7Lkv53lrZjliLA6ICVzIiwgb3V0cHV0X3BhdGgpCiAgICBwcmludChmInNhdmVkIHN1Ym1pc3Npb24gdG8ge291dHB1dF9wYXRofSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
}

for _name, _b64 in _FILES.items():
    (PKG_DIR / _name).write_text(
        base64.b64decode(_b64).decode('utf-8'),
        encoding='utf-8',
    )
    print(f"  写入 {_name}")

import sys
if str(PKG_DIR.parent) not in sys.path:
    sys.path.insert(0, str(PKG_DIR.parent))

print("\narena_ranker 已写入:", PKG_DIR)
print("sys.path 已更新")

In [ ]:
import arena_ranker
print('导入成功:', arena_ranker.__file__)

## 3. 配置

设置竞赛数据路径和训练超参数。

⚠️ **请根据你的实际竞赛修改 `COMPETITION_SLUG`。**

In [ ]:
import torch

# ============================================================
# 🔧 修改这里
# ============================================================
COMPETITION_SLUG = "llm-classification-finetuning"   # ← 改成你的竞赛 slug
MODEL_NAME       = "Qwen/Qwen3-Embedding-0.6B"      # 基础模型
EPOCHS           = 1
BATCH_SIZE       = 2        # T4 16GB 可以用 2~4
GRAD_ACCUM_STEPS = 4
MAX_LENGTH       = 512
LEARNING_RATE    = 2e-5
USE_LORA         = True
# ============================================================

DATA_DIR    = f"/kaggle/input/{COMPETITION_SLUG}"
OUTPUT_DIR  = "/kaggle/working/artifacts/default"

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
print(f"设备: {device_name} ({vram_gb:.1f} GB)")
print(f"数据目录: {DATA_DIR}")
print(f"输出目录: {OUTPUT_DIR}")


In [ ]:
import os

data_files = os.listdir(DATA_DIR)
print("竞赛数据文件:", data_files)
assert "train.csv" in data_files, f"找不到 train.csv，请检查 COMPETITION_SLUG 是否正确。当前数据目录: {DATA_DIR}"
assert "test.csv"  in data_files, f"找不到 test.csv，请检查 COMPETITION_SLUG 是否正确。当前数据目录: {DATA_DIR}"
print("✅ 数据验证通过")


## 4. 训练

使用 LoRA 微调 Qwen Embedding 模型 + MLP 分类头。

Kaggle T4 (16GB) 相比本地 8GB 显卡有更多余量，默认 `batch_size=2`，训练速度更快。

| 参数 | 默认值 | 说明 |
| --- | --- | --- |
| `BATCH_SIZE` | 2 | T4 可用 2~4，P100 可用 2~4 |
| `GRAD_ACCUM_STEPS` | 4 | 有效 batch = BATCH_SIZE × GRAD_ACCUM_STEPS |
| `MAX_LENGTH` | 512 | 输入序列最大 token 数 |
| `EPOCHS` | 1 | 训练轮数 |

In [ ]:
import sys

sys.argv = [
    "arena-train",
    "--data-dir",        DATA_DIR,
    "--output-dir",      OUTPUT_DIR,
    "--model-name",      MODEL_NAME,
    "--epochs",          str(EPOCHS),
    "--batch-size",      str(BATCH_SIZE),
    "--grad-accum-steps", str(GRAD_ACCUM_STEPS),
    "--max-length",      str(MAX_LENGTH),
]

if USE_LORA:
    sys.argv.append("--use-lora")
else:
    sys.argv.append("--disable-lora")

from arena_ranker.train import main as train_main
train_main()


## 5. 推理与生成提交文件

加载训练好的 checkpoint，对 `test.csv` 进行推理，生成 `submission.csv`。

In [ ]:
import importlib, sys

SUBMISSION_PATH = "/kaggle/working/submission.csv"

sys.argv = [
    "arena-predict",
    "--checkpoint-dir", OUTPUT_DIR,
    "--data-dir",       DATA_DIR,
    "--output-path",    SUBMISSION_PATH,
]

from arena_ranker.predict import main as predict_main
predict_main()


## 6. 检查提交文件

In [ ]:
import pandas as pd

sub = pd.read_csv(SUBMISSION_PATH)
print(f"行数: {len(sub)}")
print(f"列名: {list(sub.columns)}")
print()
print(sub.head(10))
print()
print("各列概率统计:")
print(sub.describe())
print()

row_sums = sub[["winner_model_a", "winner_model_b", "winner_tie"]].sum(axis=1)
print(f"概率行和范围: [{row_sums.min():.6f}, {row_sums.max():.6f}]")
print("✅ submission.csv 已生成:", SUBMISSION_PATH)


## 附录：离线模式（无需联网）

如果你的 notebook 不能联网（例如最终提交时），需要提前将模型上传到 Kaggle。

### 步骤

1. **上传模型到 Kaggle**
   - 在本地下载好 `Qwen/Qwen3-Embedding-0.6B` 的完整文件
   - 前往 [kaggle.com/models](https://kaggle.com/models) → New Model
   - 上传模型文件夹
   - 或者使用 Kaggle Datasets 上传也可以

2. **在 notebook 中添加模型数据集**
   - 右侧 Add Input → 搜索你上传的模型
   - 模型路径一般为 `/kaggle/input/<model-dataset-slug>/`

3. **修改配置**
   ```python
   MODEL_NAME = "/kaggle/input/<model-dataset-slug>"  # 改为本地路径
   ```

4. **训练时加上 `--local-files-only`**
   ```python
   sys.argv.append("--local-files-only")
   ```

这样 notebook 就不需要联网下载模型了。

## 附录：将代码上传为 Kaggle Dataset（替代方案）

如果你不想在 notebook 里内联所有代码，可以把整个仓库上传为 Kaggle Dataset：

1. 在本地项目目录下，打包源码：
   ```bash
   # 确保在项目根目录
   zip -r arena-ranker-code.zip src/ pyproject.toml README.md
   ```

2. 上传到 Kaggle Datasets

3. 在 notebook 中安装：
   ```python
   !pip install /kaggle/input/arena-ranker-code/
   ```

4. 然后就可以直接使用命令行工具：
   ```python
   !arena-train --data-dir /kaggle/input/llm-classification-finetuning/ --output-dir /kaggle/working/artifacts/default
   !arena-predict --checkpoint-dir /kaggle/working/artifacts/default --data-dir /kaggle/input/llm-classification-finetuning/ --output-path /kaggle/working/submission.csv
   ```

这种方式代码更整洁，且保持与本地仓库同步。